# 1. Loading an ensemble of FALL3D forecasts
***

Results from FALL3D simulations are stored in [netCDF][netcdf] format

In order to open and manipulate data from netCDF files, we can use the python package `xarray`. In order to create maps from the data, a popular option is `matplotlib` along with `Cartopy`, a Python package designed for geospatial data processing in order to produce maps and other geospatial data analyses.

Some example of Python scripts can be found in the [FALL3D user guide][fall3d].

<div class="alert alert-block alert-info">
<b>NetCDF file format:</b> 
    
NetCDF (network Common Data Form) is an open-source file format and associated software libraries for creating, accessing, and sharing scientific data, particularly multidimensional arrays like meteorological variables
</div>

[netcdf]: https://www.unidata.ucar.edu/software/netcdf "netCDF"
[fall3d]: https://fall3d-suite.gitlab.io/fall3d/hands-on/introduction.html "FALL3D user guide"

## Importing modules

In [ ]:
import xarray as xr                              # multi-dimensional arrays for NetCDF data
import numpy as np                               # numerical operations on arrays
import matplotlib.pyplot as plt                  # plots and visualizations
import cartopy.crs as crs                        # coordinate systems for maps
import cartopy.feature as cfeature

## Loading data and inspecting file structure

The N-dimensional nature of Xarray’s data structures makes it suitable for dealing with multi-dimensional scientific data, and its use of dimension names instead of axis labels (`dim='time'` instead of `axis=0`) makes such arrays much more manageable.

Here is an example of how we might structure a dataset for a weather forecast:

![](https://docs.xarray.dev/en/stable/_images/dataset-diagram.png)

In [ ]:
# Open a netCDF file in a xarray dataset
fname = "data/final.ens.nc"
ds = xr.open_dataset(fname)

# Let's see the structure of the data
ds

## Selecting variables and indexing

In [ ]:
# Selecting a single variable
da = ds['tephra_col_mass_mean']
da

In [ ]:
# Indexing
da = ds['tephra_col_mass_mean'].isel(time=-1)
da

In [ ]:
da.time

## Saving data

In [ ]:
## Saving a single variable for the last time step
da.to_netcdf("colmass.nc")

## Creating an empty map

In [ ]:
from helper_plot import create_map
fig, ax = create_map()

In [ ]:
fig, ax = create_map()
###
### Set map limits
###
ax.set_extent([-10, 15, 30, 50]) # [x1,x2,y1,y2]
###
### Volcà del Montsacopa
####
vlat, vlon = 42.187193, 2.488525
###
### Add vent location
###
ax.plot(vlon,vlat,color='red',marker='^')

## Plotting filled contours from FALL3D data

In [ ]:
fig, ax = create_map()

fc = ax.contourf(
    da.lon,da.lat,da,
    levels    = [1, 5, 10, 15, 20, 25, 30, 35, 40],
    cmap      = plt.cm.RdYlBu_r,
    extend    = 'max',
    )

cbar=fig.colorbar(
    fc, 
    orientation = 'horizontal',
    label = r'Tephra column mass [$g~m^{-2}$]',
    shrink = 0.5,                  
)
ax.set(title='Ensemble mean')

## Exercise: Exceedance probabilities

<div class="alert alert-block alert-info">
<b>Exercise:</b> Given an ensemble of column mass forecasts:
    
* Compute exceedance probabilities for an intensity threshold of 10 g/m2
* Plot contour lines for probabilities of 2%, 25%, 50%, 75%, 98%
</div>


In [ ]:
threshold = 10

da = ds['tephra_col_mass'].isel(time=-1)
probabilities = 100*(da>threshold).mean(dim='ens')
probabilities

In [ ]:
fig, ax = create_map()

cs = ax.contour(
    da.lon,da.lat,probabilities,
    levels    = [2, 25, 50, 75, 98],
    #cmap      = plt.cm.RdYlBu_r,
    #extend    = 'max',
    #transform = proj
    )
ax.clabel(cs, cs.levels, inline =False, fontsize=10)
ax.set_extent([-22, -11, 23.5, 29]) # [x1,x2,y1,y2]
ax.set(title='Exceedance probabilities')